In [1]:
import shutil
import sys
from pathlib import Path
from typing import List

sys.path.append("../src")
from validator import SimpleFileValidator


def test_validate() -> bool:
    """Чистый тест - показывает только ошибки"""

    print("Тестирование FileValidator")
    print("=" * 30)

    test_dir: Path = Path("test_temp")
    test_dir.mkdir(exist_ok=True)

    errors: List[str] = []

    # 1. Валидный PDF
    valid_pdf: Path = test_dir / "valid.pdf"
    valid_pdf.write_bytes(b"%PDF-\ntest\n%%EOF")
    is_valid: bool
    msg: str
    is_valid, msg = SimpleFileValidator.validate_file(str(valid_pdf))
    if not is_valid:
        errors.append(f"Валидный PDF не прошел: {msg}")

    # 2. Не-PDF файл
    txt_file: Path = test_dir / "not_pdf.txt"
    txt_file.write_text("текст")
    is_valid, msg = SimpleFileValidator.validate_file(str(txt_file))
    if is_valid:
        errors.append("Текстовый файл прошел как PDF")

    # 3. Пустой файл
    empty_pdf: Path = test_dir / "empty.pdf"
    empty_pdf.write_bytes(b"")
    is_valid, msg = SimpleFileValidator.validate_file(str(empty_pdf))
    if is_valid:
        errors.append("Пустой файл прошел")

    # 4. Несуществующий файл
    is_valid, msg = SimpleFileValidator.validate_file("несущ.pdf")
    if is_valid:
        errors.append("Несуществующий файл прошел")

    # 5. Испорченный PDF
    bad_pdf: Path = test_dir / "bad.pdf"
    bad_pdf.write_bytes(b"NOT PDF")
    is_valid, msg = SimpleFileValidator.validate_file(str(bad_pdf))
    if is_valid:
        errors.append("Испорченный PDF прошел")

    # 6. Слишком большой файл
    large_pdf: Path = test_dir / "large.pdf"
    with open(large_pdf, "wb") as f:
        f.write(b"%PDF-")
        f.write(b"X" * (11 * 1024 * 1024 - 5))
    is_valid, msg = SimpleFileValidator.validate_file(str(large_pdf))
    if is_valid:
        errors.append("Слишком большой файл прошел")

    # Очистка
    shutil.rmtree(test_dir)

    # Результаты
    if errors:
        print("\nОШИБКИ:")
        for error in errors:
            print(f"  - {error}")
        print(f"\nВсего ошибок: {len(errors)}")
        return False
    else:
        print("\nВсе тесты пройдены успешно")
        return True


# Запуск
test_validate()

Тестирование FileValidator

Все тесты пройдены успешно


True